# Placement 2, v2: controls, mirrored pairs and targeted JLens

This run follows up the v1 run (`notebooks/placement_2_v1.ipynb`, documented in `docs/placement_2_v1/`). v1 found that base Gemma 2 9B prefers SWERVE on every question, so v2 first checks whether that is a property of the model or of the prompt format, then runs the Jacobian Lens and interventions only on the cases that were informative in v1. Settings are in `configs/placement_2_v2.yaml`; v1 files are not changed.

| Reviewer priority | What this notebook does | Where |
| --- | --- | --- |
| 1. Behavioural controls | Gemma 2 9B Instruct with its chat template; neutral A/B labels (2 label mappings × 2 orders × 2 languages × 40 = 320 variants); greedy generated answer, A/B probability and invalid-response rate; base model kept as the baseline | B3–B6 |
| 2. Mirrored pairs | `counterbalance_pair_id` on UTILITARIAN_005/006 and 007/008 (question files v6_2, same text as v6_1); human and model pair analysis | A2.1, B6 |
| 2. Respondent-level uncertainty | user-clustered intervals and the repeated-respondent check | A2.2 |
| 2. Translation review | side-by-side English/Persian sheet with empty reviewer columns | A2.3 |
| 3. JLens | calibration on count-substituted dilemma prompts (no study configuration), approved concept lists with a coverage table, half-A/half-B repeat fit merged into the main lens | A2.4, B7 |
| 4. Interventions | 4 chosen cases, both languages and both option orders, 2 layers, 7 directions (3 controls), strengths ±0.2 and ±0.4 of the residual norm; probability change, answer change and output validity | B9 |
| 5. Reproducibility | model commit hashes, package versions, TF32 flag, v1 comparison, per-stage times | B2, B5, B11 |

**Part A2 (CPU)** runs where the repo is; only A2.2 needs `data_raw_output/`. **Part B2 (GPU)** is self-contained and writes to `output/placement_2_v2/`.

### Expected GPU time (A100 80 GB, not yet measured)

| Stage | Work | Estimate |
| --- | --- | --- |
| B4 Instruct model | load; 160 STAY/SWERVE scores + 480 short generations | 4–6 min |
| B5 Base model | load; 160 STAY/SWERVE + 320 A/B scores, one forward each | 2–3 min |
| B7 JLens fit | 6 + 6 calibration prompts (`jlens.fit_prompts_max`) | 5–8 min |
| B8 Readouts | 8 cases × 2 languages | < 1 min |
| B9 Interventions | 4 cases × 2 languages × 2 orders × 2 layers × 29 edits | 2–5 min |
| Total | | about 20–25 min |

v1 was slow because every forward ran in plain float32 and every label had its own forward. v2 turns on TF32 matrix multiplies (weights stay float32), scores each prompt with one forward, and asks the model only for the logits it needs. There are no time limits: every stage runs at the size set in the config, so reduce `jlens.fit_prompts_max`, the cases, layers, directions or strengths there to shorten the run. Each stage prints its time.

## Corrections to statements in the v1 notebook

These sentences in `notebooks/placement_2.ipynb` / `placement_2_v1.ipynb` were out of date for the v6_1 questions; `placement_2.ipynb` now has the corrected text.

| Section | v1 text | Correct |
| --- | --- | --- |
| 1.9 | "The 40 study questions all use the pedestrians, no-signal layout." | The questions use three layouts: pedestrians with no signal, pedestrians at crossings with a green and a red signal, and passengers behind a barrier against pedestrians. |
| 2.3 | "The v6_1 questions have no exact mirrors" | UTILITARIAN_005/006 and UTILITARIAN_007/008 are exact mirrors (same two groups, paths swapped). v6_1 had no `counterbalance_pair_id`, so the table was empty; v6_2 adds the IDs. |
| 2.5 | "Every pedestrian-vs-pedestrian, no-barrier, no-signal configuration" | The catalog lists every configuration in the sample in all three layouts. |

# Part A2: human-side checks (CPU)

In [ ]:
from pathlib import Path
import json

import pandas as pd
import yaml
from IPython.display import display

from multilingual_mechinterp.placement2 import calibration as cal
from multilingual_mechinterp.placement2 import controls as ct
from multilingual_mechinterp.placement2 import human_data as hd
from multilingual_mechinterp.placement2 import prompts as pr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load((ROOT / "configs" / "placement_2_v2.yaml").read_text(encoding="utf-8"))
DATA_DIR = ROOT / "data_raw_output"
HUMAN_OUT = ROOT / CFG["run"]["output_root"] / "human"
HUMAN_OUT.mkdir(parents=True, exist_ok=True)
NEAR_TIE = CFG["human"]["near_tie_warning_fraction"]

QUESTIONS_EN = pr.load_jsonl(ROOT / CFG["inputs"]["questions_en"])
QUESTIONS_FA = pr.load_jsonl(ROOT / CFG["inputs"]["questions_fa"])
REFERENCES = pd.read_csv(ROOT / CFG["inputs"]["human_references"])
assert not pr.validate_persian(QUESTIONS_EN, QUESTIONS_FA), pr.validate_persian(QUESTIONS_EN, QUESTIONS_FA)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
print(len(QUESTIONS_EN), "questions;", "pairs:",
      sorted({q["counterbalance_pair_id"] for q in QUESTIONS_EN if q.get("counterbalance_pair_id")}))
print("output:", HUMAN_OUT)

## A2.1 Mirrored pairs (humans)

UTILITARIAN_005 (STAY kills 2 pedestrians, SWERVE kills 5 passengers) and UTILITARIAN_006 (STAY kills 5 passengers, SWERVE kills 2 pedestrians) are the same dilemma with the paths swapped; so are UTILITARIAN_007 and 008 (5 pedestrians against 4 passengers). Averaging over a pair holds the groups fixed: `pct_stay_paired` is the preference for not intervening (50 = none) and `pct_expected_group_paired` is the preference for the larger group with the action held fixed. The second table gives the share sparing the larger group in each orientation.

In [ ]:
paired = hd.paired_inaction(REFERENCES, QUESTIONS_EN)
paired.to_csv(HUMAN_OUT / "paired_inaction.csv", index=False)
display(paired.style.format({c: "{:.1f}" for c in paired.columns if c.startswith(("pct", "stay_ci", "expected_ci"))})
        .hide(axis="index"))

empty = pd.DataFrame(columns=["model", "format", "question_id", "language", "p_swerve"])
human_pairs = ct.mirrored_pairs(empty, QUESTIONS_EN, REFERENCES).rename(columns={"format": "cluster"})
human_pairs.to_csv(HUMAN_OUT / "mirrored_pairs_human.csv", index=False)
display(human_pairs[["pair_id", "cluster", "questions", "p_target_first", "p_target_second",
                     "group_preference", "inaction_preference"]].round(3))

## A2.2 Respondent-level uncertainty and repeated respondents

The Wilson intervals in the reference table treat each matched response as independent. Here responses are grouped by user, and the interval uses the cluster-robust standard error, so a user who met the same configuration several times counts once. `design_effect` above 1 means the Wilson interval was too narrow; `label_changes` marks references whose label would change under the clustered interval. Needs `data_raw_output/`; skipped otherwise.

In [ ]:
if not DATA_DIR.exists():
    print(f"{DATA_DIR} not found: run this cell where the human sample lives.")
else:
    responses = hd.load_responses(DATA_DIR)
    sides = hd.action_sides(responses)
    del responses
    repeats = hd.user_repeat_check(sides, QUESTIONS_EN)
    repeats.to_csv(HUMAN_OUT / "user_repeat_check.csv", index=False)
    print("questions with any repeated user:", int((repeats.users_with_repeats > 0).sum()), "of", len(repeats))
    display(repeats.describe().loc[["mean", "max"]].round(2))

    uncertainty = hd.respondent_level_uncertainty(sides, QUESTIONS_EN, near_tie=NEAR_TIE)
    uncertainty.to_csv(HUMAN_OUT / "respondent_level_uncertainty.csv", index=False)
    per_cluster = uncertainty.groupby("cluster", sort=False).agg(
        questions=("question_id", "size"),
        median_design_effect=("design_effect", "median"),
        max_design_effect=("design_effect", "max"),
        max_abs_user_weighting_shift=("p_stay", lambda s: (s - uncertainty.loc[s.index, "p_stay_user_weighted"]).abs().max()),
        labels_changed=("label_changes", "sum"),
    )
    display(per_cluster.round(4))
    changed = uncertainty[uncertainty.label_changes]
    if len(changed):
        display(changed[["question_id", "cluster", "n_responses", "n_users", "p_stay", "label_wilson",
                         "label_cluster_robust", "design_effect"]].round(3))

## A2.3 English–Persian translation review sheet

One row per question and part (scenario text, STAY line, SWERVE line, instruction), with the English and Persian text side by side and the review topics the English part touches (passenger, pedestrian, signal, casualty wording). The reviewer fills in `meaning_matches`, `issue`, `suggested_persian`, `reviewer` and `review_date`. An existing sheet is never overwritten, so reviewer edits are safe.

In [ ]:
REVIEW_PATH = ROOT / "data" / "placement_2" / "review" / "translation_review_v6_2.csv"
if REVIEW_PATH.exists():
    review = pd.read_csv(REVIEW_PATH, keep_default_na=False)
    print(f"existing sheet: {len(review)} rows, {int((review.reviewer != '').sum())} reviewed")
else:
    REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
    review = pd.DataFrame(pr.translation_review_rows(QUESTIONS_EN, QUESTIONS_FA))
    review.to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")
    print("wrote", REVIEW_PATH.relative_to(ROOT), len(review), "rows")
topics = review.review_topics.str.split(", ").explode().value_counts()
display(topics)
display(review[review.question_id == "UTILITARIAN_005"][["part", "english", "persian", "review_topics"]])

## A2.4 JLens calibration prompts

v1 fitted the lens on Placement 1 culture questions, which are not moral dilemmas. v2 uses dilemma prompts made from the study's own English and Persian questions by changing the head counts only (for example "two people" → "four people"). No new wording is written. Count words are replaced only when they come right before a person or animal noun, so "two possible routes", "دو مسیر" and "یک خودروی خودران" stay as they are. Counts stay between 2 and 5, the larger group stays larger, and every configuration is checked against the 40 study questions. Every row carries the flag `AUTO_GENERATED_COUNT_SUBSTITUTION_OF_STUDY_TEMPLATE`.

Limitation: the prompts share the study questions' wording, so the lens is fitted on the same template, though never on the same configuration.

In [ ]:
CAL_PATH = ROOT / CFG["inputs"]["jlens_calibration"]
REGENERATE_CALIBRATION = False
if REGENERATE_CALIBRATION or not CAL_PATH.exists():
    rows = cal.count_substitution_prompts(QUESTIONS_EN, QUESTIONS_FA)
    with CAL_PATH.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("wrote", CAL_PATH.relative_to(ROOT))
cal_rows = pr.load_jsonl(CAL_PATH)
by_id = {q["question_id"]: q for q in QUESTIONS_EN}
study = {cal.signature(q) for q in QUESTIONS_EN}
clash = [r["calibration_id"] for r in cal_rows if cal.signature(by_id[r["source_question_id"]], *r["new_counts"]) in study]
assert not clash, f"calibration rows reuse a study configuration: {clash}"
table = pd.DataFrame(cal_rows)
print("no calibration configuration is a study configuration")
display(table.groupby(["split", "language"]).size().unstack(fill_value=0))
display(table[table.split != "spare"][["calibration_id", "split", "language", "source_question_id", "scenario_type",
                                       "original_counts", "new_counts"]])

# Part B2: model stages (GPU)

Start here on the GPU server. Needs:

1. The repo (with `dist/` wheel or `src/`), the human reference CSV from Part A, and the v6_2 question files.
2. Base Gemma 2 9B at `/models/gemma-2-9b` and Gemma 2 9B Instruct at `/models/gemma-2-9b-it`.
3. `data/placement_2/inputs/jlens_calibration.jsonl` and `concepts.json` (approved).

Each stage saves JSON under `output/placement_2_v2/records/` and reloads it on a rerun unless its `FORCE_*` flag is set; scoring resumes from where it stopped. Lens checkpoints (`.pt`) are kept.

In [ ]:
from pathlib import Path
import os
import runpy


def _resolve_setup_script() -> Path:
    try:
        here = Path.cwd()
    except FileNotFoundError:
        for fallback in ("/workspace/multilingual-mechinterp", "/workspace", "/root", "/"):
            if Path(fallback).is_dir():
                os.chdir(fallback)
                break
        here = Path.cwd()
    candidates = [
        here / "runpod_setup.py",
        here / "notebooks" / "runpod_setup.py",
        here.parent / "notebooks" / "runpod_setup.py",
        Path("/workspace/multilingual-mechinterp/notebooks/runpod_setup.py"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("runpod_setup.py not found. Start the kernel in the repo or in notebooks/.")


_setup = runpy.run_path(str(_resolve_setup_script()))
globals().update({k: _setup[k] for k in _setup["EXPORTS"]})

import gc
import json
import time
import traceback
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
import yaml
from IPython.display import display

import multilingual_mechinterp
from multilingual_mechinterp.jlens import JacobianLens, fit, from_hf
from multilingual_mechinterp.models.hooks import hooked_forward
from multilingual_mechinterp.models.loader import LoadedModel
from multilingual_mechinterp.patching import patch_residuals
from multilingual_mechinterp.placement2 import behavior as bh
from multilingual_mechinterp.placement2 import choices as ch
from multilingual_mechinterp.placement2 import controls as ct
from multilingual_mechinterp.placement2 import jlens_analysis as jl
from multilingual_mechinterp.placement2 import prompts as pr
from multilingual_mechinterp.placement2.experiment import (
    environment_info,
    git_info,
    model_identity,
    pending_approvals,
    sha256_file,
)
from multilingual_mechinterp.utils import ensure_dir, save_json

GPU_T0 = time.perf_counter()
CFG = yaml.safe_load((ROOT / "configs" / "placement_2_v2.yaml").read_text(encoding="utf-8"))
PLACE = ensure_dir(ROOT / CFG["run"]["output_root"])
REC = ensure_dir(PLACE / "records")
FIG = ensure_dir(PLACE / "figures")
JLENS_DIR = ensure_dir(PLACE / "jlens")

if CFG["numerics"]["allow_tf32"]:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "font.size": 11})
print("ROOT   ", ROOT)
print("OUTPUT ", PLACE)
print("package", getattr(multilingual_mechinterp, "__version__", None))
print("TF32   ", torch.backends.cuda.matmul.allow_tf32)
print("approvals still pending:", pending_approvals(CFG))

## B1 Run controls

Values come from `configs/placement_2_v2.yaml`.

In [ ]:
SEED = int(CFG["run"]["seed"])
LANGUAGES = list(CFG["behavior"]["languages"])
ORDERS = list(CFG["choices"]["presentation_orders"])
ANSWER_PREFIX = CFG["choices"]["answer_prefix"]
SEPARATOR = CFG["choices"]["continuation_separator"]
LABELS = tuple(CFG["choices"]["labels"])
NEUTRAL_LABELS = tuple(CFG["choices"]["neutral_labels"])
MAPPINGS = list(CFG["choices"]["neutral_mappings"])
NEUTRAL_ORDERS = list(CFG["choices"]["neutral_orders"])
MAX_LENGTH = 1024
JSD_THRESHOLD = float(CFG["behavior"]["high_jsd_threshold"])
P_THRESHOLD = float(CFG["behavior"]["high_p_difference_threshold"])
N_BOOT = int(CFG["behavior"]["bootstrap_samples"])

BASE_PATH = CFG["model"]["local_path"]
IT_PATH = CFG["instruct_model"]["local_path"]
GEN_TOKENS = int(CFG["instruct_model"]["generation_max_new_tokens"])

READOUT_IDS = list(CFG["cases"]["readout"])
INTERVENTION_IDS = list(CFG["cases"]["intervention"])

JLENS_LAYERS = list(CFG["jlens"]["source_layers"])
JLENS_TARGET = int(CFG["jlens"]["target_layer"])
N_FIT_MAX = int(CFG["jlens"]["fit_prompts_max"])
N_HOLDOUT = int(CFG["jlens"]["holdout_prompts"])
JLENS_DIM_BATCH = int(CFG["jlens"]["dimension_batch"])
JLENS_MAX_SEQ = int(CFG["jlens"]["max_fit_tokens"])
JLENS_SKIP_FIRST = int(CFG["jlens"]["skip_first_tokens"])
TOP_K_RAW = int(CFG["jlens"]["top_k_raw"])
TOP_K_FILTERED = int(CFG["jlens"]["top_k_filtered"])

INT_LAYERS = list(CFG["interventions"]["layers"])
INT_DIRECTIONS = list(CFG["interventions"]["directions"])
STRENGTHS = [float(s) for s in CFG["interventions"]["relative_residual_norm_strengths"]]
INT_ORDERS = list(CFG["interventions"]["orders"])

RUN_INSTRUCT = True
RUN_GENERATION = True
RUN_BASE = True
RUN_JLENS_FIT = True
RUN_READOUTS = True
RUN_INTERVENTIONS = True

FORCE_INSTRUCT = False
FORCE_BASE = False
FORCE_JLENS_FIT = False
FORCE_READOUTS = False
FORCE_INTERVENTIONS = False

torch.manual_seed(SEED)
STATUS = json.loads((REC / "status.json").read_text(encoding="utf-8")) if (REC / "status.json").exists() else {}
STAGE_TIMES = (json.loads((REC / "stage_times.json").read_text(encoding="utf-8"))
               if (REC / "stage_times.json").exists() else {})

## B2 Helpers

In [ ]:
def dump(name, obj, quiet=False):
    path = save_json(obj, REC / f"{name}.json")
    if not quiet:
        print("saved", path)
    return path


def load_record(name):
    path = REC / f"{name}.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


def as_records(frame):
    return json.loads(frame.to_json(orient="records"))


def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("saved", path)


def elapsed_min():
    return (time.perf_counter() - GPU_T0) / 60


class timed:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        print(f"--- {self.name} (notebook clock {elapsed_min():.1f} min) ---")
        return self

    def __exit__(self, exc_type, exc, tb):
        dt = time.perf_counter() - self.t0
        gpu_gb = round(torch.cuda.max_memory_allocated() / 1024**3, 2) if torch.cuda.is_available() else None
        STAGE_TIMES[self.name] = {"seconds": round(dt, 2), "peak_gpu_gb": gpu_gb}
        save_json(STAGE_TIMES, REC / "stage_times.json")
        print(f"--- {self.name} {'failed' if exc_type else 'ok'} in {dt:.1f}s  peak_gpu_gb={gpu_gb}  "
              f"notebook clock {elapsed_min():.1f} min ---")
        return False


def write_status(stage, value):
    STATUS[stage] = value
    save_json(STATUS, REC / "status.json")


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_rows(name, items, key, score, force=False, every=40):
    # Score every item not already in records/<name>.json; saves as it goes so a rerun resumes.
    saved = None if force else load_record(name)
    rows = {} if saved is None else {key(r): r for r in saved}
    todo = [item for item in items if key(item) not in rows]
    t0 = time.perf_counter()
    for i, item in enumerate(todo, start=1):
        rows[key(item)] = score(item)
        if i % every == 0 or i == len(todo):
            dump(name, list(rows.values()), quiet=True)
            rate = (time.perf_counter() - t0) / i
            print(f"  {name}: {i}/{len(todo)}  {rate:.2f}s each, about {rate * (len(todo) - i) / 60:.1f} min left")
    dump(name, list(rows.values()), quiet=True)
    return list(rows.values())


def row_key(r):
    return (r["format"], r["question_id"], r["language"], r["order"], r.get("mapping"))


def item_key(x):
    # Saved rows are dicts; items to score are (format, variant) pairs.
    return row_key(x) if isinstance(x, dict) else row_key({"format": x[0], **x[1]})


def action_row(model_tag, v, out):
    return {"model": model_tag, "format": "stay_swerve", "question_id": v["question_id"], "language": v["language"],
            "order": v["order"], "mapping": None, "swerve_first": v["order"] == "SWERVE_first",
            "p_swerve": out["p_swerve"], "p_label_A": None, "logp": out["logp"],
            "first_token_mass": out["first_token_mass"], "top1_is_label_start": out["top1_is_label_start"],
            "n_context_tokens": out["n_context_tokens"]}


def neutral_row(model_tag, v, p_by_label, first_token_mass, top1, n_context):
    return {"model": model_tag, "format": "neutral_ab", "question_id": v["question_id"], "language": v["language"],
            "order": v["order"], "mapping": v["mapping"], "swerve_first": v["swerve_first"],
            "swerve_label": v["swerve_label"], "p_swerve": p_by_label[v["swerve_label"]],
            "p_label_A": p_by_label["A"], "first_token_mass": first_token_mass, "top1_is_label_start": top1,
            "n_context_tokens": n_context}


def variant(qid, lang, order="STAY_first"):
    for row in VARIANTS:
        if (row["question_id"], row["language"], row["order"]) == (qid, lang, order):
            return row["prompt"]
    raise KeyError((qid, lang, order))

## B3 Prompts

160 STAY/SWERVE variants (40 questions × 2 languages × 2 option orders, as in v1) and 320 neutral variants. In a neutral variant the option text is unchanged; the labels become A and B, under both mappings (A = STAY, A = SWERVE) and both orders (A listed first, B listed first), and the instruction asks for "A or B". Averaging over the four variants cancels both a preference for the letter A and a preference for the first-listed option.

In [ ]:
EN_PATH = ROOT / CFG["inputs"]["questions_en"]
FA_PATH = ROOT / CFG["inputs"]["questions_fa"]
REF_PATH = ROOT / CFG["inputs"]["human_references"]

QUESTIONS_EN = pr.load_jsonl(EN_PATH)
QUESTION_BY_ID = {q["question_id"]: q for q in QUESTIONS_EN}
PERSIAN = pr.load_jsonl(FA_PATH)
problems = pr.validate_persian(QUESTIONS_EN, PERSIAN)
if problems:
    write_status("prompts", "blocked")
    raise RuntimeError(f"Persian file problems: {problems[:10]}")
if not REF_PATH.exists():
    write_status("prompts", "blocked")
    raise FileNotFoundError(f"{REF_PATH} is missing. Run Part A or copy the file from where Part A ran.")
REFERENCES = pd.read_csv(REF_PATH)

VARIANTS = pr.build_variants(QUESTIONS_EN, PERSIAN, languages=LANGUAGES, orders=ORDERS)
NEUTRAL = pr.build_neutral_variants(QUESTIONS_EN, PERSIAN, languages=LANGUAGES, mappings=MAPPINGS, orders=NEUTRAL_ORDERS)
assert len(VARIANTS) == 40 * len(LANGUAGES) * len(ORDERS)
assert len(NEUTRAL) == 40 * len(LANGUAGES) * len(MAPPINGS) * len(NEUTRAL_ORDERS)
dump("prompts", {
    "languages": LANGUAGES, "orders": ORDERS, "answer_prefix_base": ANSWER_PREFIX, "labels": list(LABELS),
    "neutral_labels": list(NEUTRAL_LABELS), "mappings": MAPPINGS, "neutral_orders": NEUTRAL_ORDERS,
    "sha256": {"questions_en": sha256_file(EN_PATH), "questions_fa": sha256_file(FA_PATH),
               "human_references": sha256_file(REF_PATH)},
    "variants": VARIANTS, "neutral_variants": NEUTRAL,
})
write_status("prompts", "ok")
print(f"{len(VARIANTS)} STAY/SWERVE variants, {len(NEUTRAL)} A/B variants")
for lang in LANGUAGES:
    example = next(v for v in NEUTRAL if v["language"] == lang and v["mapping"] == "A_is_SWERVE" and v["order"] == "A_first")
    print(f"\n----- {lang}, A = SWERVE, A listed first -----\n{example['prompt']}")

## B4 Instruction-tuned model (Gemma 2 9B Instruct, float32)

Each prompt is sent as one user turn in Gemma's chat template, and the model's reply starts right after the template's model-turn marker.

- **STAY/SWERVE format:** both labels are scored by the full continuation (summed log-probability of their tokens at the start of the reply), renormalised over the two.
- **A/B format:** A and B are single tokens, so their probabilities come from the first step of generation.
- **Generated answer:** a greedy reply of up to `GEN_TOKENS` tokens is parsed. Leading spaces, markdown and quotes are ignored, and the reply must start with a label as a whole word. Anything else is invalid and counted in `invalid_rate`.

The model is unloaded at the end of the stage so the base model fits.

In [ ]:
IT_ROWS = load_record("scores_instruct")
need = len(VARIANTS) + len(NEUTRAL)
if not RUN_INSTRUCT:
    write_status("instruct", "skipped")
elif IT_ROWS is not None and len(IT_ROWS) >= need and not FORCE_INSTRUCT:
    print(f"loaded scores_instruct.json ({len(IT_ROWS)} rows)")
    write_status("instruct", "loaded")
else:
    it = None
    try:
        with timed("instruct"):
            it = load_experiment_model(IT_PATH, dtype=CFG["instruct_model"]["dtype"])
            it.model.eval()
            tok = it.tokenizer
            IT_ENV = {
                "model_name": it.name, "dtype": str(next(it.model.parameters()).dtype), "n_layers": it.n_layers,
                "model_identity": model_identity({"model": {**CFG["instruct_model"], "local_path": IT_PATH}}),
                "label_tokens": {l: tok(l, add_special_tokens=False)["input_ids"] for l in (*LABELS, *NEUTRAL_LABELS)},
                "chat_example": ct.chat_text(tok, "PROMPT"),
            }
            print(json.dumps({k: IT_ENV[k] for k in ("model_name", "dtype", "label_tokens", "chat_example")}, indent=1))
            AB_IDS = ct.label_token_ids(tok, NEUTRAL_LABELS, "")
            assert AB_IDS is not None, "A and B must be single tokens at the start of the reply"
            fn = ch.hf_logits_fn(it.model)

            def score_action(v):
                context = ct.chat_text(tok, v["prompt"])
                out = ch.score_prompt(fn, tok, context, answer_prefix="", separator="", labels=LABELS,
                                      max_length=MAX_LENGTH, add_special_tokens=False)
                row = action_row("instruct", v, out)
                if RUN_GENERATION:
                    text, _, _ = ct.generate_answer(it.model, tok, context, max_new_tokens=GEN_TOKENS, max_length=MAX_LENGTH)
                    parsed = ct.parse_answer(text, LABELS)
                    row.update(generated_text=text, parsed=parsed, parsed_action=parsed, valid=parsed is not None)
                return row

            def score_neutral(v):
                context = ct.chat_text(tok, v["prompt"])
                text, first, n_context = ct.generate_answer(it.model, tok, context, max_length=MAX_LENGTH,
                                                            max_new_tokens=GEN_TOKENS if RUN_GENERATION else 1)
                two = ct.two_way_from_row(first, AB_IDS)
                row = neutral_row("instruct", v, two["p"], two["first_token_mass"], two["top1_is_label_start"], n_context)
                if RUN_GENERATION:
                    parsed = ct.parse_answer(text, NEUTRAL_LABELS)
                    action = None if parsed is None else ("SWERVE" if parsed == v["swerve_label"] else "STAY")
                    row.update(generated_text=text, parsed=parsed, parsed_action=action, valid=parsed is not None)
                return row

            items = [("stay_swerve", v) for v in VARIANTS] + [("neutral_ab", v) for v in NEUTRAL]
            IT_ROWS = run_rows(
                "scores_instruct", items, key=item_key,
                score=lambda x: score_action(x[1]) if x[0] == "stay_swerve" else score_neutral(x[1]),
                force=FORCE_INSTRUCT,
            )
            dump("environment_instruct", IT_ENV)
        write_status("instruct", "ok")
    except Exception:
        write_status("instruct", "failed")
        traceback.print_exc()
    finally:
        it = fn = None
        free_gpu()
        if torch.cuda.is_available():
            print(f"GPU memory after unloading: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## B5 Base model (Gemma 2 9B, float32)

The same prompts with the v1 scoring: prompt + `ANSWER_PREFIX`, continuations ` STAY` / ` SWERVE` and ` A` / ` B`. Base models do not follow instructions, so there is no generation; `top1_is_label_start` records whether the most likely next token starts one of the labels.

**Reproducibility:** the STAY/SWERVE scores are compared with the v1 `scores.json` when it is available. Small differences are expected from TF32 and from single-forward scoring. A decision change would need investigating.

In [ ]:
if USE_TINY_OFFLINE:
    raise RuntimeError("Placement 2 needs Gemma 2 9B. USE_TINY_OFFLINE must stay False.")

model = load_experiment_model(BASE_PATH, dtype=CFG["model"]["dtype"])
if not isinstance(model, LoadedModel):
    raise TypeError(f"Expected LoadedModel, got {type(model).__name__}")
model.model.eval()
D_MODEL = int(model.model.config.hidden_size)
ENV = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_name": model.name,
    "n_layers": model.n_layers,
    "d_model": D_MODEL,
    "dtype": str(next(model.model.parameters()).dtype),
    "device": str(model.device),
    "allow_tf32": bool(torch.backends.cuda.matmul.allow_tf32),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "package": getattr(multilingual_mechinterp, "__version__", None),
    "model_identity": model_identity({**CFG, "model": {**CFG["model"], "local_path": BASE_PATH}}),
    "instruct_model": load_record("environment_instruct"),
    "git": git_info(ROOT),
    "environment": environment_info(),
    "seed": SEED,
    "config_sha256": sha256_file(ROOT / "configs" / "placement_2_v2.yaml"),
    "scoring_base": "sum of continuation-token log-probs after prompt + answer prefix, renormalised over the two labels",
    "scoring_instruct": "chat template; full continuation for STAY/SWERVE, first-step probabilities for A/B; greedy reply parsed",
    "hook": "blocks.{layer} output (resid_post); edits overwrite the last context position",
}
dump("environment", ENV)
print(json.dumps({k: ENV[k] for k in ("model_name", "dtype", "allow_tf32", "transformers")}, indent=1))
print("local commits:", ENV["model_identity"].get("local_commit"),
      (ENV["instruct_model"] or {}).get("model_identity", {}).get("local_commit"))


def score_text(prompt, edit=None, labels=LABELS):
    fn = ch.hf_logits_fn(model.model, edit=edit)
    return ch.score_prompt(fn, model.tokenizer, prompt, answer_prefix=ANSWER_PREFIX, separator=SEPARATOR,
                           labels=labels, max_length=MAX_LENGTH)


def score_base(item):
    kind, v = item
    if kind == "stay_swerve":
        return action_row("base", v, score_text(v["prompt"]))
    out = score_text(v["prompt"], labels=NEUTRAL_LABELS)
    return neutral_row("base", v, {"A": out["p_a"], "B": out["p_b"]}, out["first_token_mass"],
                       out["top1_is_label_start"], out["n_context_tokens"])


BASE_ROWS = load_record("scores_base")
if not RUN_BASE:
    write_status("base", "skipped")
elif BASE_ROWS is not None and len(BASE_ROWS) >= len(VARIANTS) + len(NEUTRAL) and not FORCE_BASE:
    print(f"loaded scores_base.json ({len(BASE_ROWS)} rows)")
    write_status("base", "loaded")
else:
    try:
        with timed("base"):
            print("label tokens:", {l: model.tokenizer(SEPARATOR + l, add_special_tokens=False)["input_ids"]
                                    for l in (*LABELS, *NEUTRAL_LABELS)})
            items = [("stay_swerve", v) for v in VARIANTS] + [("neutral_ab", v) for v in NEUTRAL]
            BASE_ROWS = run_rows("scores_base", items, key=item_key, score=score_base, force=FORCE_BASE)
        write_status("base", "ok")
    except Exception:
        write_status("base", "failed")
        traceback.print_exc()
    free_gpu()

V1_CANDIDATES = [ROOT / CFG["inputs"]["v1_scores"], ROOT / "output" / "placement_2" / "model_stages" / "records" / "scores.json"]
V1_PATH = next((p for p in V1_CANDIDATES if p.exists()), None)
REPRO = {"v1_scores": str(V1_PATH) if V1_PATH else None}
if V1_PATH and BASE_ROWS:
    new = pd.DataFrame([r for r in BASE_ROWS if r["format"] == "stay_swerve"])
    old = pd.DataFrame(json.loads(V1_PATH.read_text(encoding="utf-8")))
    REPRO.update(ct.compare_runs(new, old))
print("reproducibility against v1:", REPRO)
dump("reproducibility", REPRO)

## B6 Behaviour under the four conditions

The conditions are {base, instruct} × {STAY/SWERVE labels, A/B labels}. Per question and language, `p_swerve` averages all variants. The overview reports per condition and language:

- **Decision share and spread:** `share_swerve_decisions`, `sd_p_swerve`.
- **`consequence_sensitivity`:** mean p(SWERVE) where SWERVE spares the expected group, minus mean p(SWERVE) where STAY does. A constant action preference gives 0.
- **Biases:** `mean_position_bias` (first-listed option), `mean_p_label_A` (0.5 = no letter preference) and `mapping_flips` (questions whose decision changes with the letter mapping).
- **Validity:** `invalid_rate` (generated replies that are not a label) and `median_first_token_mass`.

Agreement with human majorities is computed per condition. The mirrored pairs show whether the model keeps sparing the larger group when the paths are swapped.

In [ ]:
ROWS = pd.DataFrame((IT_ROWS or []) + (BASE_ROWS or []))
ROWS.drop(columns=["logp"], errors="ignore").to_csv(REC / "scores_all_conditions.csv", index=False)
SUMMARY = ct.variant_summary(ROWS)
SUMMARY.to_csv(REC / "behaviour_by_question.csv", index=False)
OVERVIEW = ct.condition_overview(SUMMARY, QUESTIONS_EN)
OVERVIEW.to_csv(REC / "condition_overview.csv", index=False)
display(OVERVIEW.round(3))

comparisons = []
for (m, f), sub in SUMMARY.groupby(["model", "format"], sort=False):
    comp = bh.language_comparison(sub, "en", "fa", jsd_threshold=JSD_THRESHOLD, p_threshold=P_THRESHOLD)
    comparisons.append(comp.assign(model=m, format=f))
COMPARISON = pd.concat(comparisons, ignore_index=True)
COMPARISON.to_csv(REC / "language_comparison.csv", index=False)
groups = COMPARISON.pivot_table(index=["model", "format"], columns="group", values="question_id", aggfunc="count", fill_value=0)
print("English-Persian case groups per condition (D1 = EN STAY / FA SWERVE, D2 = EN SWERVE / FA STAY):")
display(groups)

In [ ]:
aligns, align_summaries, correlations = [], [], []
for (m, f), sub in SUMMARY.groupby(["model", "format"], sort=False):
    table = bh.alignment_table(sub, REFERENCES).assign(model=m, format=f)
    aligns.append(table)
    align_summaries.append(bh.alignment_summary(table, n_boot=N_BOOT, seed=SEED).assign(model=m, format=f))
    for (lang, cluster), g in table.groupby(["language", "cluster"], sort=False):
        correlations.append({"model": m, "format": f, "language": lang, "cluster": cluster,
                             "pearson_r": g.p_model_swerve.corr(g.p_human_swerve),
                             "spearman_r": jl.spearman(g.p_model_swerve, g.p_human_swerve)})
ALIGN = pd.concat(aligns, ignore_index=True)
ALIGN.to_csv(REC / "alignment_by_question.csv", index=False)
ALIGN_SUMMARY = pd.concat(align_summaries, ignore_index=True).merge(
    pd.DataFrame(correlations), on=["model", "format", "language", "cluster"])
ALIGN_SUMMARY.to_csv(REC / "alignment_summary.csv", index=False)
display(ALIGN_SUMMARY[ALIGN_SUMMARY.cluster == "All clusters"][
    ["model", "format", "language", "agreement_labelled", "agreement_labelled_ci_low", "agreement_labelled_ci_high",
     "n_labelled", "mean_abs_error", "pearson_r", "spearman_r"]].round(3))

PAIRS = ct.mirrored_pairs(SUMMARY, QUESTIONS_EN, REFERENCES)
PAIRS.to_csv(REC / "mirrored_pairs.csv", index=False)
display(PAIRS[["pair_id", "source", "model", "format", "language", "p_target_first", "p_target_second",
               "group_preference", "inaction_preference", "target_spared_both"]].round(3))

if "valid" in ROWS:
    gen = ROWS[ROWS.valid.notna()]
    validity = gen.groupby(["format", "language"]).agg(variants=("valid", "size"),
                                                       invalid_rate=("valid", lambda s: 1 - s.astype(float).mean()),
                                                       median_first_token_mass=("first_token_mass", "median"))
    validity.to_csv(REC / "generation_validity.csv")
    display(validity.round(3))
    invalid = gen[~gen.valid.astype(bool)]
    if len(invalid):
        print("examples of invalid replies:")
        display(invalid[["format", "question_id", "language", "order", "mapping", "generated_text"]].head(12))
    agree = gen[gen.valid.astype(bool)].assign(
        scored=lambda t: np.where(t.p_swerve > 0.5, "SWERVE", "STAY"))
    print("generated answer equals the higher-probability label:",
          f"{(agree.scored == agree.parsed_action).mean():.3f} of {len(agree)} valid replies")

In [ ]:
conditions = list(OVERVIEW[["model", "format"]].drop_duplicates().itertuples(index=False, name=None))
expected = {q["question_id"]: q["expected_action"] for q in QUESTIONS_EN}
fig, axes = plt.subplots(1, len(conditions), figsize=(4 * len(conditions), 4), sharey=True, squeeze=False)
for ax, (m, f) in zip(axes[0], conditions):
    sub = SUMMARY[(SUMMARY.model == m) & (SUMMARY.format == f)].assign(expected=lambda t: t.question_id.map(expected))
    for j, (lang, marker) in enumerate((("en", "o"), ("fa", "s"))):
        for i, action in enumerate(("STAY", "SWERVE")):
            vals = sub[(sub.language == lang) & (sub.expected == action)].p_swerve
            x = i + (j - 0.5) * 0.25 + np.random.default_rng(i + 10 * j).uniform(-0.05, 0.05, len(vals))
            ax.scatter(x, vals, marker=marker, alpha=0.7, label=lang if i == 0 else None)
    ax.axhline(0.5, color="gray", lw=0.8)
    ax.set_xticks([0, 1], ["STAY spares\nexpected group", "SWERVE spares\nexpected group"])
    ax.set_title(f"{m}, {f}")
    ax.set_ylim(0, 1)
axes[0][0].set_ylabel("p(SWERVE), all variants averaged")
axes[0][0].legend()
fig.suptitle("Does p(SWERVE) follow the consequences? (a constant preference gives two equal clouds)")
savefig(fig, "01_consequence_sensitivity.png")

pair_ids = sorted(PAIRS.pair_id.unique())
fig, axes = plt.subplots(1, len(pair_ids), figsize=(7 * len(pair_ids), 4), squeeze=False)
for ax, pid in zip(axes[0], pair_ids):
    sub = PAIRS[(PAIRS.pair_id == pid) & ((PAIRS.source == "model") | (PAIRS.format == "All clusters"))]
    names = [("humans (all clusters)" if r.source == "human" else f"{r.model} {r.format} {r.language}") for r in sub.itertuples()]
    x = np.arange(len(sub))
    ax.bar(x - 0.2, sub.p_target_first, 0.4, label=sub.questions.iloc[0].split(" / ")[0])
    ax.bar(x + 0.2, sub.p_target_second, 0.4, label=sub.questions.iloc[0].split(" / ")[1])
    ax.axhline(0.5, color="gray", lw=0.8)
    ax.set_xticks(x, names, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("p(sparing the larger group)")
    ax.set_ylim(0, 1)
    ax.set_title(pid)
    ax.legend(fontsize=8)
savefig(fig, "02_mirrored_pairs.png")
print(f"notebook clock {elapsed_min():.1f} min")

## B7 Jacobian Lens (base model)

**Concept coverage.** Only words that are a single Gemma token are scored. The table lists, per family and language, how many words survive and which are dropped. A Persian family with no surviving word falls back to the English words for the interventions, and this is recorded.

**Fit.** The calibration prompts are split into two halves, each with both languages, and fitted alternately one prompt at a time: half A prompt 1, half B prompt 1, half A prompt 2, and so on. The main lens is the merge of the two halves, so the repeat-fit stability check costs no extra GPU time. Each prompt is checkpointed, so an interrupted fit resumes. The number of fit prompts is `jlens.fit_prompts_max` in the config.

**Stability on the 4 holdout prompts**, half A vs half B:
- top-50 token overlap;
- top-8 content-token overlap;
- rank correlation of concept scores;
- relative Jacobian difference `‖J_A − J_B‖ / ‖J‖`;
- cosine between the two halves' back-projected direction for each concept family (the directions used in B9).

In [ ]:
CONCEPTS = jl.load_concepts(ROOT / CFG["inputs"]["concepts"])
CONCEPT_STATUS = CONCEPTS.get("status", "draft")
if CONCEPT_STATUS != "approved":
    raise RuntimeError(f"concepts.json status is {CONCEPT_STATUS!r}; v2 needs an approved list")
FAMILY_IDS, FAMILY_DROPPED = {}, {}
coverage = []
for lang in ("en", "fa"):
    FAMILY_IDS[lang], FAMILY_DROPPED[lang] = jl.family_token_ids(model.tokenizer, CONCEPTS["families"], lang)
    for family, words in CONCEPTS["families"].items():
        listed = words.get(lang, [])
        coverage.append({"family": family, "language": lang, "words": len(listed),
                         "single_token_words": len(listed) - len(FAMILY_DROPPED[lang][family]),
                         "token_ids": len(FAMILY_IDS[lang][family]),
                         "dropped": ", ".join(FAMILY_DROPPED[lang][family]),
                         "falls_back_to_english": lang == "fa" and not FAMILY_IDS[lang][family]})
COVERAGE = pd.DataFrame(coverage)
COVERAGE.to_csv(REC / "concept_coverage.csv", index=False)
display(COVERAGE)

LENS_MODEL = from_hf(model.model, model.tokenizer, force_bos=False)
CAL_ROWS = pr.load_jsonl(ROOT / CFG["inputs"]["jlens_calibration"])
FIT_ROWS = [r for r in CAL_ROWS if r["split"] == "fit"][:N_FIT_MAX]
HOLDOUT_ROWS = [r for r in CAL_ROWS if r["split"] == "holdout"][:N_HOLDOUT]
HALVES = {"half_a": [r["prompt"] for i, r in enumerate(FIT_ROWS) if (i // 2) % 2 == 0],
          "half_b": [r["prompt"] for i, r in enumerate(FIT_ROWS) if (i // 2) % 2 == 1]}
HOLDOUT_PROMPTS = [r["prompt"] for r in HOLDOUT_ROWS]
lens_paths = {name: JLENS_DIR / f"{name}.pt" for name in ("jacobian_lens", "jacobian_lens_half_a", "jacobian_lens_half_b")}
ckpt_paths = {half: JLENS_DIR / f"{half}_fit_checkpoint.pt" for half in HALVES}
print("calibration:", {k: len(v) for k, v in HALVES.items()}, "holdout", len(HOLDOUT_PROMPTS),
      "| languages in fit rows:", pd.Series([r["language"] for r in FIT_ROWS]).value_counts().to_dict())


def lens_logits(lens, text, positions):
    logits, _, _ = lens.apply(LENS_MODEL, text, layers=JLENS_LAYERS, positions=positions,
                              max_seq_len=MAX_LENGTH, use_jacobian=True)
    return logits


def top_tokens(row, k):
    values, indices = torch.topk(row, k=k)
    return [{"token_id": int(i), "token": model.tokenizer.decode([int(i)]), "logit": float(v)}
            for v, i in zip(values.tolist(), indices.tolist())]


LENS = LENS_A = LENS_B = None
if not RUN_JLENS_FIT:
    write_status("jlens_fit", "skipped")
elif all(p.exists() for p in lens_paths.values()) and not FORCE_JLENS_FIT:
    LENS, LENS_A, LENS_B = (JacobianLens.load(str(lens_paths[n])) for n in lens_paths)
    print(f"loaded lenses: half A {LENS_A.n_prompts}, half B {LENS_B.n_prompts}, merged {LENS.n_prompts} prompts")
    write_status("jlens_fit", "loaded")
else:
    try:
        with timed("jlens_fit"):
            if FORCE_JLENS_FIT:
                for p in [*ckpt_paths.values(), *lens_paths.values()]:
                    p.unlink(missing_ok=True)
            lenses, log = {}, []
            t_start = time.perf_counter()
            for k in range(1, max(len(p) for p in HALVES.values()) + 1):
                for half, prompts in HALVES.items():
                    if k > len(prompts):
                        continue
                    t = time.perf_counter()
                    lenses[half] = fit(LENS_MODEL, prompts[:k], source_layers=JLENS_LAYERS, target_layer=JLENS_TARGET,
                                       dim_batch=JLENS_DIM_BATCH, max_seq_len=JLENS_MAX_SEQ, skip_first=JLENS_SKIP_FIRST,
                                       checkpoint_path=str(ckpt_paths[half]), checkpoint_every=1, resume=True)
                    dt = time.perf_counter() - t
                    log.append({"half": half, "prompt": k, "seconds": round(dt, 1)})
                    print(f"  {half} prompt {k}: {dt:.1f}s  (fit clock {(time.perf_counter() - t_start) / 60:.1f} min)")
            LENS_A, LENS_B = lenses["half_a"], lenses["half_b"]
            LENS = JacobianLens.merge([LENS_A, LENS_B])
            for name, lens in (("jacobian_lens", LENS), ("jacobian_lens_half_a", LENS_A), ("jacobian_lens_half_b", LENS_B)):
                lens.save(str(lens_paths[name]))
            print(f"saved lenses: half A {LENS_A.n_prompts}, half B {LENS_B.n_prompts}, merged {LENS.n_prompts} prompts")

            W = model.model.get_output_embeddings().weight.detach().float().cpu()
            stability = {"calibration_source": CFG["inputs"]["jlens_calibration"], "n_half_a": LENS_A.n_prompts,
                         "n_half_b": LENS_B.n_prompts, "n_holdout": len(HOLDOUT_PROMPTS),
                         "fit_log": log, "holdout": [], "layers": {}}
            all_ids = {**FAMILY_IDS["en"], **{f"fa:{k}": v for k, v in FAMILY_IDS["fa"].items()}}
            for index, prompt in enumerate(HOLDOUT_PROMPTS):
                a_logits, b_logits = lens_logits(LENS_A, prompt, [-1]), lens_logits(LENS_B, prompt, [-1])
                entry = {"holdout_index": index, "layers": {}}
                for layer in JLENS_LAYERS:
                    a, b = a_logits[layer][-1], b_logits[layer][-1]
                    raw_a, raw_b = top_tokens(a, TOP_K_RAW), top_tokens(b, TOP_K_RAW)
                    sa, sb = jl.family_scores(a, all_ids), jl.family_scores(b, all_ids)
                    keys = [k for k in all_ids if sa[k] is not None]
                    entry["layers"][str(layer)] = {
                        "half_raw_top50_jaccard": jl.jaccard([t["token_id"] for t in raw_a], [t["token_id"] for t in raw_b]),
                        "half_content_top8_jaccard": jl.jaccard(
                            [t["token"].strip() for t in jl.content_tokens(raw_a, TOP_K_FILTERED)],
                            [t["token"].strip() for t in jl.content_tokens(raw_b, TOP_K_FILTERED)]),
                        "half_family_score_spearman": jl.spearman([sa[k] for k in keys], [sb[k] for k in keys]),
                    }
                stability["holdout"].append(entry)
            for layer in JLENS_LAYERS:
                ja, jb, jf = LENS_A.jacobians[layer].float(), LENS_B.jacobians[layer].float(), LENS.jacobians[layer].float()
                cosines = {}
                for family, ids in FAMILY_IDS["en"].items():
                    if ids:
                        da, db = jl.backprojected_direction(ja, W, ids), jl.backprojected_direction(jb, W, ids)
                        cosines[family] = float(torch.dot(da, db))
                stability["layers"][str(layer)] = {
                    **{key: float(np.mean([e["layers"][str(layer)][key] for e in stability["holdout"]
                                           if e["layers"][str(layer)][key] is not None]))
                       for key in ("half_raw_top50_jaccard", "half_content_top8_jaccard", "half_family_score_spearman")},
                    "jacobian_relative_difference": float((ja - jb).norm() / jf.norm()),
                    "direction_cosine_min": min(cosines.values()),
                    "direction_cosine_mean": float(np.mean(list(cosines.values()))),
                    "direction_cosine_by_family": cosines,
                }
            dump("jlens_stability", stability)
            display(pd.DataFrame({l: {k: v for k, v in d.items() if k != "direction_cosine_by_family"}
                                  for l, d in stability["layers"].items()}).T.round(3))
        write_status("jlens_fit", "ok")
    except Exception:
        write_status("jlens_fit", "failed")
        traceback.print_exc()
    free_gpu()

## B8 Readouts on the eight cases

For each case and language (STAY listed first, prompt plus answer prefix), the lens is read at the last token of the `STAY:` line, the last token of the `SWERVE:` line, and the answer position. Saved per layer: the raw top 50 tokens, the top 8 after removing template and answer-format tokens, and concept-family scores (log P(family), with English and with Persian words). `concept_contrast` at the answer position = score(family STAY kills) − score(family SWERVE kills).

In [ ]:
READOUTS = load_record("jlens_readouts")
POSITION_NAMES = ("STAY", "SWERVE", "last")
if not RUN_READOUTS:
    write_status("jlens_readouts", "skipped")
elif LENS is None and not lens_paths["jacobian_lens"].exists():
    print("Need jlens/jacobian_lens.pt.")
    write_status("jlens_readouts", "blocked")
elif READOUTS is not None and not FORCE_READOUTS:
    print(f"loaded jlens_readouts.json ({len(READOUTS)} rows)")
    write_status("jlens_readouts", "loaded")
else:
    try:
        with timed("jlens_readouts"):
            if LENS is None:
                LENS = JacobianLens.load(str(lens_paths["jacobian_lens"]))
            READOUTS = []
            for qid in READOUT_IDS:
                families = jl.question_families(QUESTION_BY_ID[qid], CONCEPTS["level_to_family"])
                wanted = [*families.values(), CONCEPTS["unrelated_family"], CONCEPTS["template_family"]]
                per_lang = {}
                for lang in LANGUAGES:
                    text = variant(qid, lang) + ANSWER_PREFIX
                    pos = jl.option_line_positions(model.tokenizer, text, max_length=MAX_LENGTH)
                    names = [n for n in POSITION_NAMES if n in pos]
                    logits = lens_logits(LENS, text, [pos[n] for n in names])
                    layers_out = {}
                    for layer in JLENS_LAYERS:
                        by_pos = {}
                        for j, name in enumerate(names):
                            row = logits[layer][j]
                            raw = top_tokens(row, TOP_K_RAW)
                            entry = {"raw_top": raw, "content_top": jl.content_tokens(raw, TOP_K_FILTERED)}
                            for word_lang in ("en", "fa"):
                                ids = {f: FAMILY_IDS[word_lang].get(f, []) for f in wanted}
                                entry[f"family_scores_{word_lang}_words"] = jl.family_scores(row, ids)
                                entry[f"family_ranks_{word_lang}_words"] = jl.family_ranks(row, ids)
                            by_pos[name] = entry
                        layers_out[str(layer)] = by_pos
                    per_lang[lang] = {"positions": {n: int(pos[n]) for n in names}, "layers": layers_out}
                row_out = {"question_id": qid, "families": families, "languages": per_lang,
                           "intervention_case": qid in INTERVENTION_IDS}
                row_out["en_fa_content_jaccard"] = {
                    str(layer): {name: jl.jaccard(
                        [t["token"].strip() for t in per_lang["en"]["layers"][str(layer)][name]["content_top"]],
                        [t["token"].strip() for t in per_lang["fa"]["layers"][str(layer)][name]["content_top"]])
                        for name in POSITION_NAMES
                        if name in per_lang["en"]["layers"][str(layer)] and name in per_lang["fa"]["layers"][str(layer)]}
                    for layer in JLENS_LAYERS}
                READOUTS.append(row_out)
            dump("jlens_readouts", READOUTS)
        write_status("jlens_readouts", "ok")
    except Exception:
        write_status("jlens_readouts", "failed")
        traceback.print_exc()
    free_gpu()

if READOUTS:
    rows = []
    for r in READOUTS:
        for lang, block in r["languages"].items():
            for layer in JLENS_LAYERS:
                last = block["layers"][str(layer)].get("last", {})
                out = {"question_id": r["question_id"], "language": lang, "layer": layer,
                       "content_top": " | ".join(t["token"].strip() for t in last.get("content_top", [])[:5])}
                fam = r["families"]
                for word_lang in ("en", "fa"):
                    scores = last.get(f"family_scores_{word_lang}_words") or {}
                    if scores.get(fam["stay_kills"]) is not None and scores.get(fam["swerve_kills"]) is not None:
                        out[f"concept_contrast_{word_lang}_words"] = scores[fam["stay_kills"]] - scores[fam["swerve_kills"]]
                rows.append(out)
    READOUT_TABLE = pd.DataFrame(rows)
    READOUT_TABLE.to_csv(REC / "jlens_readout_summary.csv", index=False)
    display(READOUT_TABLE[READOUT_TABLE.layer == JLENS_LAYERS[-1]])
    pairs = READOUT_TABLE[READOUT_TABLE.question_id.isin(["UTILITARIAN_005", "UTILITARIAN_006", "UTILITARIAN_007", "UTILITARIAN_008"])]
    if "concept_contrast_en_words" in pairs:
        print("mirrored pairs: the contrast should change sign when the paths swap if the lens tracks the groups")
        display(pairs.pivot_table(index=["question_id", "language"], columns="layer", values="concept_contrast_en_words").round(3))

## B9 Interventions on four cases

Cases: UTILITARIAN_005 and 006 (a mirrored pair), AGE_006 and FITNESS_007. Each case is run in both languages and both option orders. At layers `INT_LAYERS`, the residual at the answer position is replaced by `residual + s · ||residual|| · direction`, for `s` in `STRENGTHS` and for a zero-strength check.

| Direction | Meaning |
| --- | --- |
| `jacobian_swap` | lens back-projection of (SWERVE-killed family − STAY-killed family); positive s should raise p(SWERVE) if the model weighs the groups |
| `embedding_swap` | the same contrast with output embeddings (Placement 1 method) |
| `jacobian_stay_kills` / `jacobian_swerve_kills` | each family alone |
| `random_same_norm`, `unrelated_concept`, `template_token` | controls |

Recorded per edit:
- the change in p(SWERVE);
- the decision per order, and the order-averaged decision;
- output validity: `first_token_mass`, and whether the top next token still starts a label.

An edit that drives the label tokens out of the top of the distribution is flagged `degraded`, and its probability change should not be read as a preference change.

In [ ]:
def context_residuals(prompt, layers):
    ids = ch.context_ids(model.tokenizer, prompt + ANSWER_PREFIX, max_length=MAX_LENGTH)
    x = torch.tensor([ids], device=model.device)
    with torch.no_grad(), hooked_forward(model.model, layers=[int(l) for l in layers]) as cache:
        model.model(input_ids=x)
        out = {int(l): cache.get(f"blocks.{int(l)}")[0, len(ids) - 1].detach().float().cpu() for l in layers}
    return out, len(ids) - 1


def patch_edit(layer, values, pos):
    dtype = next(model.model.parameters()).dtype
    v = values.detach().to(device=model.device, dtype=dtype).unsqueeze(0)
    p = torch.tensor([int(pos)], device=model.device)
    return lambda: patch_residuals(model.model, {int(layer): v}, p)


EDITS = load_record("interventions")
if not RUN_INTERVENTIONS:
    write_status("interventions", "skipped")
elif LENS is None and not lens_paths["jacobian_lens"].exists():
    print("Need jlens/jacobian_lens.pt.")
    write_status("interventions", "blocked")
elif EDITS is not None and not FORCE_INTERVENTIONS:
    print(f"loaded interventions.json ({len(EDITS['rows'])} rows)")
    write_status("interventions", "loaded")
else:
    try:
        with timed("interventions"):
            if LENS is None:
                LENS = JacobianLens.load(str(lens_paths["jacobian_lens"]))
            W = model.model.get_output_embeddings().weight.detach().float().cpu()
            t_start = time.perf_counter()
            rows, completed = [], []
            for qid in INTERVENTION_IDS:
                fams = jl.question_families(QUESTION_BY_ID[qid], CONCEPTS["level_to_family"])
                for lang in LANGUAGES:
                    ids = FAMILY_IDS[lang]
                    word_lang = {side: lang if ids.get(fams[side]) else "en" for side in fams}
                    tokens = {side: ids.get(fams[side]) or FAMILY_IDS["en"].get(fams[side], []) for side in fams}
                    for order in INT_ORDERS:
                        prompt = variant(qid, lang, order)
                        clean = score_text(prompt)
                        residuals, pos = context_residuals(prompt, INT_LAYERS)
                        for layer in INT_LAYERS:
                            residual, J = residuals[layer], LENS.jacobians[layer]
                            dirs = {}
                            if tokens["stay_kills"] and tokens["swerve_kills"]:
                                js, jw = (jl.backprojected_direction(J, W, tokens[s]) for s in ("stay_kills", "swerve_kills"))
                                es, ew = (jl.embedding_direction(W, tokens[s]) for s in ("stay_kills", "swerve_kills"))
                                dirs.update({"jacobian_swap": jl.unit(jw - js), "embedding_swap": jl.unit(ew - es),
                                             "jacobian_stay_kills": js, "jacobian_swerve_kills": jw})
                            dirs["random_same_norm"] = jl.random_direction(D_MODEL, SEED + layer)
                            for name, family in (("unrelated_concept", CONCEPTS["unrelated_family"]),
                                                 ("template_token", CONCEPTS["template_family"])):
                                fam_ids = ids.get(family) or FAMILY_IDS["en"].get(family)
                                if fam_ids:
                                    dirs[name] = jl.embedding_direction(W, fam_ids)
                            base = {"question_id": qid, "language": lang, "order": order, "layer": layer,
                                    "concept_word_language": word_lang, "residual_norm": float(residual.norm()),
                                    "clean_p_swerve": clean["p_swerve"], "clean_decision": clean["decision"],
                                    "clean_first_token_mass": clean["first_token_mass"],
                                    "clean_top1_is_label_start": clean["top1_is_label_start"]}
                            jobs = [("zero_strength", None, 0.0)] + [(n, dirs[n], s) for n in INT_DIRECTIONS if n in dirs for s in STRENGTHS]
                            for name, direction, s in jobs:
                                values = residual if direction is None else jl.edited_residual(residual, direction, s)
                                out = score_text(prompt, edit=patch_edit(layer, values, pos))
                                rows.append({**base, "direction": name, "strength": s, "p_swerve": out["p_swerve"],
                                             "delta_p_swerve": out["p_swerve"] - clean["p_swerve"],
                                             "decision": out["decision"], "flipped": out["decision"] != clean["decision"],
                                             "first_token_mass": out["first_token_mass"],
                                             "top1_is_label_start": out["top1_is_label_start"],
                                             "degraded": (clean["top1_is_label_start"] and not out["top1_is_label_start"])
                                             or out["first_token_mass"] < 0.5 * clean["first_token_mass"]})
                completed.append(qid)
                print(f"done {qid}  ({(time.perf_counter() - t_start) / 60:.1f} min)")
                dump("interventions", {"rows": rows, "completed_cases": completed}, quiet=True)
            EDITS = {"rows": rows, "completed_cases": completed}
            dump("interventions", EDITS)
        write_status("interventions", "ok")
    except Exception:
        write_status("interventions", "failed")
        traceback.print_exc()
    free_gpu()

In [ ]:
if EDITS and EDITS["rows"]:
    EDIT_TABLE = pd.DataFrame(EDITS["rows"])
    EDIT_TABLE.to_csv(REC / "interventions.csv", index=False)
    zero = EDIT_TABLE[EDIT_TABLE.direction == "zero_strength"]
    print(f"zero-strength check: max |Δp(SWERVE)| = {zero.delta_p_swerve.abs().max():.2e} (should be ~0)")
    print("completed cases:", EDITS["completed_cases"])

    keys = ["question_id", "language", "layer", "direction", "strength"]
    avg = EDIT_TABLE.groupby(keys).agg(p_swerve=("p_swerve", "mean"), clean_p_swerve=("clean_p_swerve", "mean"),
                                       n_orders=("order", "nunique"), any_degraded=("degraded", "any")).reset_index()
    avg["delta_p_swerve"] = avg.p_swerve - avg.clean_p_swerve
    avg["flipped_order_averaged"] = (avg.p_swerve > 0.5) != (avg.clean_p_swerve > 0.5)
    avg.to_csv(REC / "interventions_order_averaged.csv", index=False)

    EDIT_SUMMARY = avg[avg.direction != "zero_strength"].groupby(["direction", "strength", "language"]).agg(
        n=("delta_p_swerve", "size"),
        mean_delta_p_swerve=("delta_p_swerve", "mean"),
        mean_abs_delta_p_swerve=("delta_p_swerve", lambda s: s.abs().mean()),
        flip_rate_order_averaged=("flipped_order_averaged", "mean"),
        degraded_rate=("any_degraded", "mean"),
    ).reset_index()
    EDIT_SUMMARY.to_csv(REC / "intervention_summary.csv", index=False)
    display(EDIT_SUMMARY[EDIT_SUMMARY.strength.abs() == max(abs(s) for s in STRENGTHS)].round(4))

    controls = {"random_same_norm", "unrelated_concept", "template_token"}
    strongest = avg[(avg.strength.abs() == max(abs(s) for s in STRENGTHS)) & (avg.direction != "zero_strength")]
    vs = strongest.assign(kind=np.where(strongest.direction.isin(controls), "control", "targeted")).groupby(
        ["kind", "language"]).delta_p_swerve.apply(lambda s: s.abs().mean()).unstack()
    print("mean |Δp(SWERVE)| at the largest strength, targeted vs control directions:")
    display(vs.round(4))
    flips = avg[avg.flipped_order_averaged & (avg.direction != "zero_strength")]
    print(f"order-averaged decision flips: {len(flips)} of {int((avg.direction != 'zero_strength').sum())} edits "
          f"({int(flips.any_degraded.sum())} of them with degraded output)")
    if len(flips):
        display(flips.sort_values("delta_p_swerve", key=abs, ascending=False).head(20).round(3))

    names = [d for d in INT_DIRECTIONS if d in set(EDIT_SUMMARY.direction)]
    fig, axes = plt.subplots(1, len(LANGUAGES), figsize=(7 * len(LANGUAGES), 4.5), squeeze=False)
    for ax, lang in zip(axes[0], LANGUAGES):
        sub = EDIT_SUMMARY[EDIT_SUMMARY.language == lang]
        for name in names:
            line = sub[sub.direction == name].sort_values("strength")
            ax.plot(line.strength, line.mean_delta_p_swerve, "--" if name in controls else "-", marker="o", label=name)
        ax.axhline(0, color="gray", lw=0.8)
        ax.set_xlabel("strength (fraction of residual norm)")
        ax.set_ylabel("mean Δp(SWERVE), order-averaged")
        ax.set_title(f"{lang}: {len(EDITS['completed_cases'])} cases, layers {INT_LAYERS}")
        ax.legend(fontsize=7)
    savefig(fig, "03_intervention_dose_response.png")

## B10 Report notes and timing

Reads the saved records only and writes `records/report_notes.md`. The last table lists the time of each stage.

In [ ]:
def pct(x):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{100 * x:.1f}%"


def md_table(frame, digits=3):
    frame = frame.round(digits)
    cols = list(frame.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + " --- |" * len(cols)]
    out += ["| " + " | ".join(str(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return out


env = load_record("environment") or {}
lines = ["# Placement 2 v2 notes", "", "## Setup", ""]
lines += [
    f"- Base model: {env.get('model_name')} ({env.get('dtype')}), commit {env.get('model_identity', {}).get('local_commit')}",
    f"- Instruct model: {(env.get('instruct_model') or {}).get('model_name')}, commit "
    f"{(env.get('instruct_model') or {}).get('model_identity', {}).get('local_commit')}",
    f"- TF32 matmuls: {env.get('allow_tf32')}; torch {env.get('torch')}, transformers {env.get('transformers')}, package {env.get('package')}",
    f"- Questions: {CFG['inputs']['questions_en']} / {CFG['inputs']['questions_fa']}; concepts {CONCEPT_STATUS}",
    f"- Reproducibility vs v1 base scores: {json.dumps(load_record('reproducibility'))}",
    f"- Stage status: {json.dumps(STATUS)}",
    "", "## Behaviour by condition", "",
]
lines += md_table(OVERVIEW)
lines += ["", "## Mirrored pairs", ""]
lines += md_table(PAIRS[["pair_id", "source", "model", "format", "language", "p_target_first", "p_target_second",
                         "group_preference", "inaction_preference"]])
lines += ["", "## Agreement with human majorities (All clusters)", ""]
for row in ALIGN_SUMMARY[ALIGN_SUMMARY.cluster == "All clusters"].itertuples():
    lines.append(f"- {row.model} {row.format} {row.language}: agreement {pct(row.agreement_labelled)} "
                 f"[{pct(row.agreement_labelled_ci_low)}, {pct(row.agreement_labelled_ci_high)}] on {row.n_labelled} labelled; "
                 f"mean |p_model − p_human| {row.mean_abs_error:.3f}; Pearson r {row.pearson_r:.3f}")
lines += ["", "## Jacobian Lens", ""]
stab = load_record("jlens_stability") or {}
lines.append(f"- Calibration: {stab.get('calibration_source')}; half A {stab.get('n_half_a')}, half B {stab.get('n_half_b')} prompts, "
             f"{stab.get('n_holdout')} holdout")
for layer, vals in (stab.get("layers") or {}).items():
    lines.append(f"- Layer {layer}: " + json.dumps({k: (round(v, 3) if isinstance(v, float) else v)
                                                   for k, v in vals.items() if k != "direction_cosine_by_family"}))
lines.append(f"- Concept words dropped (multi-token): " + json.dumps(
    {f"{r.language}:{r.family}": r.dropped for r in COVERAGE.itertuples() if r.dropped}, ensure_ascii=False))
lines += ["", "## Interventions", ""]
if EDITS and "EDIT_SUMMARY" in globals():
    lines.append(f"- Completed cases: {EDITS['completed_cases']}")
    top = EDIT_SUMMARY[EDIT_SUMMARY.strength.abs() == max(abs(s) for s in STRENGTHS)].sort_values("mean_abs_delta_p_swerve", ascending=False)
    lines += md_table(top)
lines += ["", "## Limitations", ""]
for item in (
    "Calibration prompts reuse the study wording with different head counts; they never repeat a study configuration.",
    "Human references pool respondents of the balanced cluster sample; see human/respondent_level_uncertainty.csv for user-clustered intervals.",
    "Agreement counts only labelled human majorities; UNDETERMINED references are excluded and nothing is scored as right or wrong.",
    "JLens readouts and interventions use the base model; the instruct model is used for behaviour only.",
    "The translation review sheet is generated for a bilingual reviewer; its conclusions are not part of this run.",
):
    lines.append(f"- {item}")
notes = REC / "report_notes.md"
notes.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("saved", notes)

timing = pd.DataFrame([{"stage": k, "minutes": v["seconds"] / 60, "peak_gpu_gb": v["peak_gpu_gb"]} for k, v in STAGE_TIMES.items()],
                      columns=["stage", "minutes", "peak_gpu_gb"])
display(timing.round(2))
print(f"notebook clock: {elapsed_min():.1f} min; stage total {timing.minutes.sum():.1f} min")